In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
import random
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from onnx2torch import convert
from skl2onnx.helpers.onnx_helper import load_onnx_model
from tqdm import tqdm

from service.finetune.finetune_after_stitching_chest_xray import (
    finetune_after_stitching,
)


2025-11-24 21:24:22.671264715 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [3]:
batch_size = 32
val_batch_size = 64
num_epochs = 3
directory = f"../_results_chest_xray/evaluation_after_finetuning/finetune_{num_epochs}"
os.makedirs(directory, exist_ok=True)


In [4]:
def save_figure(x, y, xlabel, ylabel, title, model_name, filename, directory):
    fig, ax = plt.subplots()
    ax = sns.lineplot(x=x, y=y, ax=ax)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.figure.savefig(f"{directory}/{model_name}/{filename}.png")
    plt.close()


In [5]:
random.seed(50)
np.random.seed(24)
torch.manual_seed(77)


In [6]:
def patch_reduce_modules(model):
    """Patch OnnxReduceStaticAxes modules to fix forward() signature issue."""
    try:
        from onnx2torch.node_converters.reduction import OnnxReduceStaticAxes
    except ImportError:
        # If import fails, try alternative import path
        try:
            from onnx2torch.converter.node_converters.reduction import (
                OnnxReduceStaticAxes,
            )
        except ImportError:
            # If still fails, return model unchanged
            return model

    def patch_module(module):
        """Recursively patch all OnnxReduceStaticAxes modules."""
        for child_name, child_module in list(module.named_children()):
            if isinstance(child_module, OnnxReduceStaticAxes):
                # Store the original forward method
                original_forward = child_module.forward

                # Create a wrapper that accepts extra arguments but ignores them
                def make_fixed_forward(orig_forward):
                    def fixed_forward(input_tensor, *args, **kwargs):
                        # OnnxReduceStaticAxes.forward should only take input_tensor
                        # but it's being called with extra args, so we ignore them
                        return orig_forward(input_tensor)

                    return fixed_forward

                # Replace the forward method with the fixed version
                child_module.forward = make_fixed_forward(original_forward)
            else:
                # Recursively patch child modules
                patch_module(child_module)

    patch_module(model)
    return model


def load_torch_model(model_path, target_opset=13):
    """Load ONNX model, patch metadata/attributes for onnx2torch compatibility, then convert."""
    model_onnx = load_onnx_model(model_path)

    # Try different opset versions in descending order
    for opset_version in [18, 13, 11]:
        try:
            model_copy = load_onnx_model(model_path)  # Reload fresh copy

            # Clamp opset version
            for opset in model_copy.opset_import:
                if opset.domain in ("", "ai.onnx") and opset.version > opset_version:
                    opset.version = opset_version

            # Fix Reshape allowzero attribute
            for node in model_copy.graph.node:
                if node.op_type == "Reshape":
                    for attr in node.attribute:
                        if attr.name == "allowzero" and attr.i == 1:
                            attr.i = 0

            # Try conversion
            torch_model = convert(model_copy)

            # Patch the converted model to fix OnnxReduceStaticAxes issues
            torch_model = patch_reduce_modules(torch_model)

            return torch_model
        except NotImplementedError:
            # If this opset version doesn't work, try next lower one
            continue

    # If all opset versions fail, raise the last error
    raise NotImplementedError(
        "Could not convert model with any supported opset version"
    )


In [7]:
latest_result_dir = (
    "../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5"
)
model_paths = sorted(glob(f"{latest_result_dir}/*.onnx"))

print(f"Fine-tuning models from: {latest_result_dir}")
print(f"Found {len(model_paths)} models to fine-tune")

best_accuracy = 0
best_model_name = ""

for index, model_path in tqdm(enumerate(model_paths), position=0, leave=True):
    print(model_path)
    model_name = model_path.split("/")[-1].replace(".onnx", "")
    print(model_name)
    if int(model_name[3:]) < 232:
        continue

    try:
        os.makedirs(f"{directory}/{model_name}", exist_ok=False)
    except OSError:
        continue

    model = None
    try:
        model = load_torch_model(model_path)
    except (NotImplementedError, Exception) as e:
        print(f"Skipping {model_name} (conversion failed): {e}")
        continue

    assert model is not None, "Model should be defined at this point"
    try:
        train_acc_history, train_loss_history, final_accuracy = (
            finetune_after_stitching(
                model,
                num_classes=2,  # NORMAL, PNEUMONIA
                batch_size=batch_size,
                num_epochs=num_epochs,
                val_batch_size=val_batch_size,
                feature_extracting=False,
            )
        )

        if final_accuracy > best_accuracy:
            best_accuracy = final_accuracy
            best_model_name = model_name

        with open(f"{directory}/{model_name}/{final_accuracy}.txt", "w") as f:
            for idx, (loss, accuracy) in enumerate(
                zip(train_loss_history, train_acc_history)
            ):
                f.write(f"Epoch: {idx + 1}, Loss: {loss}, Accuracy: {accuracy}\n")

        save_figure(
            x=range(len(train_acc_history)),
            y=train_acc_history,
            xlabel="epoch",
            ylabel="accuracy",
            title=f"{model_name} accuracy data",
            model_name=model_name,
            filename="accuracy",
            directory=directory,
        )
        save_figure(
            x=range(len(train_loss_history)),
            y=train_loss_history,
            xlabel="epoch",
            ylabel="loss",
            title=f"{model_name} loss data",
            model_name=model_name,
            filename="loss",
            directory=directory,
        )

        model = model.cpu()
        torch.onnx.export(
            model,
            torch.ones(1, 3, 224, 224),
            f"{directory}/{model_name}/model_ft.onnx",
            opset_version=18,
            dynamo=False,
        )
        del model
        torch.cuda.empty_cache()
    except (TypeError, RuntimeError, Exception) as e:
        print(f"Skipping {model_name} (training failed): {e}")
        if "model" in locals():
            del model
        torch.cuda.empty_cache()
        continue

with open(f"{directory}/final_result.txt", "w") as f:
    f.write(f"Best model is {best_model_name} with accuracy {best_accuracy}")

print("\n✓ Fine-tuning complete!")
print(f"Best model: {best_model_name} with accuracy: {best_accuracy}")


Fine-tuning models from: ../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5
Found 234 models to fine-tune


234it [00:00, 166153.23it/s]

../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net000.onnx
net000
../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net001.onnx
net001
../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net002.onnx
net002
../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net003.onnx
net003
../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net004.onnx
net004
../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net005.onnx
net005
../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net006.onnx
net006
../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net007.onnx
net007
../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net008.onnx
net008
../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net009.onnx
net009
../_results_chest_xray/1763902492_result_BS_32_MD_16_T_0_TT_0.5_K_5/net010.onnx
net010
../_results_chest_xray/1763902492_result_BS